# Real M/EEG is messy — and the noise is never the same shape

### There is no universally correct denoiser. `mne-denoise` matches the method to the structure of the contamination, and lets you audit what it did.

```
        PERIODIC              TRANSIENT           REFERENCE-CORRELATED        A TARGET YOU DECLARE
     power-line noise      movement bursts        recorded noise channels      reproducible response
            │                     │                        │                            │
            ▼                     ▼                        ▼                            ▼
   ZapLine / SpectrumInterp      ASR                   iCanClean                       DSS
```

Every act below asks one question, runs the real estimator, and reports **two**
numbers: did the artifact go down, and did the neural signal survive.

MNE-Python maintainers sprint &nbsp;·&nbsp; Meta Paris

## 0 — Setup

On Colab this installs a pinned `mne-denoise` and fetches a 20 MB data bundle.
Locally it is a no-op — everything is already cached.

In [ ]:
# Colab / fresh-environment bootstrap. Does nothing on a prepared machine.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEMO_REPO = "https://github.com/snesmaeili/mne-denoise-meta-demo.git"
MNE_DENOISE_PIN = "f5b821cc2a535e84ed46085d45ea5a356dd8d548"
MNE_DENOISE_SPEC = (
    f"mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@{MNE_DENOISE_PIN}"
)

def _pip(spec):
    # subprocess, not %pip: line magics do not interpolate {braces}, so an
    # f-string here is the only way the pin actually reaches pip.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)

if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(["git", "clone", "-q", "--depth", "1", DEMO_REPO],
                          capture_output=True, text=True)
    if done.returncode != 0:
        print("Could not clone the demo repository.")
        print("git said:", (done.stderr or "").strip() or done.returncode)
        raise SystemExit(
            "If the repository is still private, the Colab VM has no credentials "
            "for it -- authorising Colab lets it OPEN a notebook, not clone the "
            "repo. Make the repository public, or run this notebook locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())

try:
    import mne_denoise  # noqa: F401
except ImportError:
    _pip(MNE_DENOISE_SPEC)

# Pull the prepared assets only if the cache is incomplete. The existence check
# is deliberately cheap so a prepared machine pays nothing here.
import os

_cache = Path(os.environ.get("MNE_DENOISE_META_DEMO_CACHE",
                             Path.home() / ".cache" / "mne-denoise" / "meta-demo"))
if not (_cache / "zapline_metrics.json").exists():
    subprocess.run([sys.executable, "fetch_demo_data.py"], check=False)
else:
    print(f"demo assets already present in {_cache}")

In [ ]:
PRESENTER_MODE = True      # stage settings: quiet, fast, no network
LIVE = True                # False -> load the cached result instead of computing it
RECOMPUTE = False          # True re-runs the slow paths instead of loading the cache
SHOW_DIAGNOSTICS = False   # extra panels, only if someone asks
RANDOM_STATE = 97

import sys, warnings, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt
import mne

import demo_utils as du

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
if PRESENTER_MODE:
    # Two narrowly-scoped filters. Warnings about rank, calibration, or any
    # algorithmic failure stay switched on deliberately.
    warnings.filterwarnings("ignore", message=".*figure layout has changed.*")
    # DSS builds its biased Epochs with mne.EpochsArray(data, info) and does not
    # carry tmin/baseline across, so MNE reports the *internal* object as
    # un-baselined. The epochs we pass in are baseline corrected -- verified in
    # README.md ("Known issues"). Filtered so a library metadata bug does not
    # look like a data problem on stage.
    warnings.filterwarnings(
        "ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline

du.assert_presenter_ready()

---
## 1 — Can two methods remove the same 50 Hz peak at very different cost?

*Mobile EEG recorded while walking a university campus — OpenNeuro `ds003620`, 32 channels, 50 Hz mains.*

In [ ]:
# The real estimator call, run live on a 60 s slice of the recording.
from mne_denoise.zapline import ZapLine
from mne_denoise.qa import noise_surround_ratio

m = du.load_json(du.cache_path("zapline_metrics.json"))
raw = mne.io.read_raw_fif(du.cache_path("zapline_demo_raw.fif"), preload=True, verbose="ERROR")

if LIVE:
    zap = ZapLine(sfreq=raw.info["sfreq"], line_freq=50.0, adaptive=True, n_select="auto")
    clean = zap.fit_transform(raw.copy())

    chunks = zap.adaptive_results_["chunk_info"]
    print(f"detected {zap.adaptive_results_['line_freq']:.0f} Hz, removed "
          f"{zap.n_removed_} component(s) over {len(chunks)} adaptive chunk(s)")

    psd_kw = dict(method="welch", fmin=1.0, fmax=125.0, n_fft=8192, verbose="ERROR")
    for name, obj in (("before", raw), ("after ", clean)):
        p = obj.compute_psd(**psd_kw)
        r = np.median(noise_surround_ratio(p.freqs, p.get_data(), 50.0, peak_bw=0.5))
        print(f"   R(50 Hz) {name} = {r:.2f}")
else:
    lc = m["live_crop"]
    print(f"(cached) 60 s window at {lc['start_s']:.0f} s, "
          f"uncorrected R(50 Hz) = {lc['window_ratio']:.2f}")

In [ ]:
# Full-recording result, prepared offline (see prepare_meta_demo.py --zapline).
m = du.load_json(du.cache_path("zapline_metrics.json"))
S = du.load_npz(du.cache_path("zapline_spectra.npz"))

with du.presentation_theme():
    fig = du.plot_line_noise_triptych(
        S["freqs"],
        {"original": S["psd_original"], "notch": S["psd_notch"], "zapline+": S["psd_zapline"]},
        line_freq=m["line_freq"], ratios=m["ratio"], fmin=40.0, fmax=65.0,
    )
plt.show()

print(f"R = residual peak power / local spectral floor      (R = 1 means 'at the floor')")
for k in ("original", "notch", "zapline+"):
    print(f"   {k:<10s} R = {m['ratio'][k]:.2f}")

> **Say:** both remove the peak. The notch drives the residual to a fifth of the surrounding
> floor — it removes signal that was never artifact. ZapLine+ lands on the floor because it
> models the *spatial* line subspace instead of imposing a fixed spectral depth.

In [ ]:
if SHOW_DIAGNOSTICS:
    A = du.load_npz(du.cache_path("zapline_adaptive.npz"))
    with du.presentation_theme():
        fig = du.plot_adaptive_component_timeline(
            A["chunk_start"], A["chunk_n_removed"],
            contamination=(A["contamination_times"], A["contamination_ratio"]),
        )
    plt.show()
    print(f"{m['n_chunks']} chunks, {m['chunks_with_zero_removed']} of them removed nothing.")
    print("Contamination over the recording spans "
          f"R = {m['contamination_range'][0]:.2f} to {m['contamination_range'][1]:.2f}.")

---
## 2 — What did ASR actually detect, and what did it cost?

*A synthetic recording where the clean signal is known exactly, plus blinks, muscle bursts,
electrode pops and a covariance shift.*

In [ ]:
F = du.load_npz(du.cache_path("asr_fixture.npz"))
a = du.load_json(du.cache_path("asr_metrics.json"))
head = a[f"{a['headline_duration_s']:.0f}s"]["standard"]

with du.presentation_theme():
    fig = du.plot_asr_reconstruction_panel(
        F["times"], F["contaminated"], F["cleaned"], F["clean"],
        repair=(F["window_times"], F["n_components_reconstructed"]),
        channels=(0, 14, 22), ch_names=list(a["headline_ch_names"]),
        tlim=(1.0, 19.0),   # display window only; metrics use the whole 60 s
    )
plt.show()

print(f"artifact intervals   RRMSE {head['artifact_rrmse_before']:.3f} "
      f"-> {head['artifact_rrmse_after']:.3f}")
print(f"artifact-free data   RRMSE {head['clean_rrmse_before']:.3f} "
      f"-> {head['clean_rrmse_after']:.3f}   <- the cost")

In [ ]:
# The fitted object says where it acted, and how much reference data it had.
print(f"windows modified            {head['fraction_windows_modified']:6.1%}")
print(f"samples repaired            {head['fraction_samples_repaired']:6.1%}")
print(f"recall on true artifacts    {head['sample_mask_recall_on_artifact']:6.1%}")
print(f"calibration                 {head['calibration_samples_per_dim']:.0f} samples "
      f"per channel dimension")

short = a["20s"]["standard"]
print(f"\nSame estimator, same defaults, a 20 s recording instead of 60 s:")
print(f"   calibration              {short['calibration_samples_per_dim']:.0f} samples/dim")
print(f"   artifact RRMSE           {short['artifact_rrmse_before']:.3f} "
      f"-> {short['artifact_rrmse_after']:.3f}")
print(f"   artifact-free RRMSE      {short['clean_rrmse_before']:.3f} "
      f"-> {short['clean_rrmse_after']:.3f}")

> **Say:** here we know the clean target, so we can price the cleaning. ASR found every
> contaminated sample. It also touched data that needed nothing — and `calibration_info_`
> tells us why: starve the calibration and both endpoints get worse together.

### The package ships four ASR variants. Which one does *this* recording need?

In [ ]:
V = du.load_json(du.cache_path("asr_variants_metrics.json"))
a, b = V["arm_a_contaminated_calibration"], V["arm_b_calibration_supply"]

with du.presentation_theme():
    fig = du.plot_asr_variant_regimes(a, b)
plt.show()

print(f"method='standard' and method='riemannian_windowed' give identical results here: "
      f"{a['methods_identical']}")
print(f"   because both already default to cov_estimator='geometric_median'")
print(f"\n{b['dataset']}")
for r in b["rows"]:
    print(f"   {r['variant']:<15s} calibrates on {100 * r['calibration_fraction']:5.1f}% "
          f"of the recording ({r['calibration_kind']}-based), {r['runtime_s']:.1f} s")

> **Say:** Blum's Riemannian robustness is real — you can see the dirty-calibration
> threshold inflate twice as much without it. But in this package it is already the
> default for every variant, so the `method=` flag changes nothing. And on this
> recording the window selector is not starving, so Juggler is not indicated. The
> fitted state answers "which variant" before you have to guess.

---
## 3 — Why do I need this, when MNE already ships Xdawn and SSD?

DSS maximises a ratio you *declare*:

$$\max_w \; \frac{w^{\top} R_{\text{biased}}\, w}{w^{\top} R_{\text{baseline}}\, w}$$

PCA, Xdawn, SSD and CSP are all this same problem with $R_{\text{biased}}$ **frozen** at one choice. DSS leaves it as an argument.

*A synthetic fixture with two planted sources, then ERP CORE N170, faces vs cars, 40 participants.*

In [ ]:
# DSS fits in a fraction of a second, so this one runs live.
from mne_denoise.dss import DSS, AverageBias

epochs = mne.read_epochs(du.cache_path("dss_demo-epo.fif"), preload=True, verbose="ERROR")
d = du.load_json(du.cache_path("dss_metrics.json"))

print(f"{len(epochs)} trials, {len(epochs.ch_names)} channels")
if LIVE:
    dss = DSS(bias=AverageBias(axis="epochs"), n_select="auto")
    dss.fit(epochs)
    print(f"DSS kept {dss.n_selected_} components; "
          f"leading bias scores {np.round(dss.eigenvalues_[:4], 3)}")
else:
    print(f"(cached) DSS kept {d['n_selected']} components; "
          f"leading bias scores {np.round(d['eigenvalues'][:4], 3)}")

# One fixture, two planted sources, three criteria. The 10 Hz rhythm is the
# STRONGER source by declaration, so variance-maximisation should return it.
b = d["bias_swap"]
amp = b["_amplitudes"]
print(f"\nplanted: evoked at amplitude {amp['evoked']}, alpha at {amp['alpha']} "
      f"(the distractor is stronger)")
print("|cos| of component 1 against each planted pattern:")
for name in ("PCA", "AverageBias", "BandpassBias"):
    print(f"   {name:<13s} evoked {b[name]['evoked']:.3f}    alpha {b[name]['alpha']:.3f}")

In [ ]:
G = du.load_json(du.cache_path("dss_group.json"))
g = G["summary"]
r = d["reproducibility"]

with du.presentation_theme():
    fig = du.plot_dss_framework_panel(b, r, group=g)
plt.show()

print("median split-half reproducibility, evaluated on held-out trials:")
for key, label in (("sensor", "raw sensors"), ("pca", "PCA, matched rank"),
                   ("dss", "DSS AverageBias"), ("xdawn", "Xdawn (already in MNE)")):
    print(f"   {label:<24s} {r[f'{key}_median']:.4f}")

print(f"\nAcross all {g['n_subjects']} participants:")
print(f"   reproducibility improved in    {g['reproducibility_gain_positive']}/{g['n_subjects']}")
print(f"   discriminability improved in   {g['auc_change_positive']}/{g['n_subjects']}")
print(f"   DSS beat matched-rank PCA in   {g['dss_over_pca_positive']}/{g['n_subjects']}")
print(f"   DSS beat Xdawn in              {g['dss_over_xdawn_positive']}/{g['n_subjects']}")
print(f"   plain PCA beat raw sensors in  {g['pca_over_sensor_positive']}/{g['n_subjects']}")

> **Say:** take the right-hand panel first, because it is the one that argues against me.
> At enhancing an evoked response, DSS does *not* beat Xdawn — Xdawn wins in 25 of 40
> participants. Plain PCA beats raw sensors in all 40. If "concentrate the evoked response"
> is your whole problem, MNE already solved it and you do not need this.
>
> The argument is the left panel. Same fixture, same estimator, one argument changed — and
> the answer moves from the evoked source to the rhythm. PCA returns the rhythm too, because
> the rhythm has more variance; it has no way to be asked for anything else. Xdawn cannot
> become SSD. That is what you are buying: the criterion is a parameter, not a fixed part of
> the algorithm.
>
> And the caution stands — reproducibility improved in 37 of 40, condition discriminability
> in only 25. Declaring the target is not the same as getting the science for free.

In [ ]:
if SHOW_DIAGNOSTICS:
    # The per-participant view behind the 37/40 and 25/40 counts: every subject
    # is one dot, reproducibility gain against condition-AUC change.
    D = du.load_npz(du.cache_path("dss_sources.npz"))
    per = G["per_subject"]
    with du.presentation_theme():
        fig = du.plot_dss_target_panel(
            D["times"],
            du.zscore_rows(D["sensor_half_a"]), du.zscore_rows(D["sensor_half_b"]),
            du.zscore_rows(D["component_half_a"]), du.zscore_rows(D["component_half_b"]),
            group={
                "reproducibility_gain": [v["r_dss"] - v["r_sensor"] for v in per.values()],
                "auc_change": [v["auc_dss"] - v["auc_sensor"] for v in per.values()],
            },
            labels={"sensor": f"sensor {d['best_channel']}", "dss": "DSS component 1"},
            window=tuple(d["n170_window_s"]),
            pattern=D["pattern"], info=epochs.info,
        )
    plt.show()

    ov = d["subspace_overlap_dss_xdawn"]
    print("DSS vs Xdawn, principal-angle cosines (1.0 = same direction):")
    print(f"   {np.round(ov, 3).tolist()}")
    print(f"   mean {np.mean(ov):.3f} -- the leading direction agrees, the rest does not.")
    print("   So Xdawn is NOT a special case of DSS as implemented; it is a different")
    print("   fixed choice of the same kind of criterion.")

---
## 4 — Does artifact attenuation mean the method worked?

*Blinks on ERP CORE N170, `sub-005` — the same participant as Act 3.*

Three ways to get a reference: the **EOG electrodes** we actually recorded, MNE's own `EOGRegression` on those same electrodes, and iCanClean's **pseudo-reference** — a filtered copy of the EEG itself, so no dedicated electrodes are needed at all.

In [ ]:
E = du.load_json(du.cache_path("eog_metrics.json"))
T = du.load_npz(du.cache_path("eog_traces.npz"))
order = [r["method"] for r in E["rows"]]

with du.presentation_theme():
    fig = du.plot_icanclean_control_panel(
        T["times"], {k: T[k] for k in order}, E["rows"],
        channel=E["blink_channel_index"], channel_name=E["blink_channel"],
    )
plt.show()

print(f"{E['n_blinks']} blinks over {E['duration_s']:.0f} s; the faces-vs-cars N170 "
      f"effect at {'/'.join(E['n170_roi'])} is {E['baseline_n170_effect_uv']:+.2f} µV "
      f"before any cleaning\n")
print(f"{'arm':<34s} {'blink removed':>13s} {'N170 effect':>12s} {'alpha':>9s}")
for r in E["rows"][1:]:
    print(f"{r['method'].replace(chr(10), ' '):<34s} {r['attenuation_pct']:12.1f}% "
          f"{r['n170_effect_uv']:+11.2f} µV {r['alpha_vs_background_db']:+8.2f} dB")

best = max(E["rows"][1:], key=lambda r: r["attenuation_pct"])
print(f"\nmost artifact removed: {best['method'].replace(chr(10), ' ')} "
      f"({best['attenuation_pct']:.1f}%) — and its N170 effect has the wrong sign "
      f"({best['n170_effect_uv']:+.2f} µV).")
print(f"posterior alpha scores it {best['alpha_vs_background_db']:+.2f} dB, which "
      f"looks fine, because 8-13 Hz is outside the band it damaged.")

> **Say:** the pseudo-reference is the appealing one — it needs no electrodes at all, you
> just notch the brain band out of the EEG and use that as the reference. And it wins on
> attenuation: 92% of the blink, against 83% for the real EOG electrodes.
>
> It also destroys the experiment. The faces-versus-cars N170 is a *negative* deflection.
> With the recorded electrodes it goes from −0.74 to −1.43 µV — removing blink noise
> sharpens it. With the pseudo-reference it comes back **positive**. The effect is not
> weakened, it is gone, because a reference built by band-filtering the EEG shares the
> signal's own low frequencies, and the N170 lives there.
>
> Now the part I want you to take away. Posterior alpha rates that same arm at −0.78 dB —
> unremarkable. It cannot see the damage, because 8 to 13 Hz is outside the band the method
> touched. If you pick your preservation endpoint outside the band you cleaned, you will
> publish this.
>
> One participant, and the pseudo-reference is the right tool when the artifact and your
> signal *don't* overlap in frequency. Here they do.

---
## 5 — You have seen two of these. The same contract covers the rest.

| Contamination structure | Method | What information it uses |
|---|---|---|
| power-line noise, possibly non-stationary | **ZapLine / ZapLine+** | narrowband spatial structure at the line frequency |
| line noise, conservative + phase-preserving | **SpectrumInterpolation** | the spectral neighbourhood of the peak |
| large transient / movement artifacts | **ASR**, AdaptiveASR, JugglerASR | abnormal covariance vs a clean baseline |
| recorded noise reference channels | **iCanClean** | correlation between scalp and reference channels |
| channel-specific sensor noise | **SNS** | what neighbouring sensors agree on |
| a target response you can define | **DSS** | a bias you declare (trial average, band, period) |

In [ ]:
with du.presentation_theme():
    fig = du.plot_contract_screen()
plt.show()

> **Say:** different assumptions, different information, different failure modes — but one
> MNE-native estimator contract, and in every case the fitted object is the thing that let us
> check the claim.

---
---

# Appendix — the ASR family in full

*Not part of the five minutes. Run these during questions or at the sprint table.*

## A1 — What ASR accepts

In [ ]:
import numpy as np, mne
from mne_denoise.asr import ASR

rng = np.random.default_rng(RANDOM_STATE)
info = mne.create_info([f"EEG{i:03d}" for i in range(8)], 250.0, "eeg")
raw_demo = mne.io.RawArray(rng.standard_normal((8, 2500)), info, verbose="ERROR")
raw_demo.filter(1.0, None, verbose="ERROR")   # ASR's documented precondition
arr = raw_demo.get_data()
epo_demo = mne.make_fixed_length_epochs(raw_demo, duration=2.0, preload=True, verbose="ERROR")

for label, obj, kw in [
    ("ndarray (n_ch, n_times)", arr, dict(sfreq=250.0)),
    ("mne.io.Raw", raw_demo, {}),
    ("mne.Epochs", epo_demo, {}),
    ("mne.Evoked", epo_demo.average(), {}),
]:
    try:
        out = ASR(**kw).fit_transform(obj if not hasattr(obj, "copy") else obj.copy())
        print(f"   {label:<26s} -> {type(out).__name__}")
    except Exception as exc:
        print(f"   {label:<26s} -> {type(exc).__name__}: {str(exc)[:58]}")

> `Evoked` is rejected for *calibration* by design — there is no within-trial variability to
> estimate a clean covariance from. A 2-D array needs `sfreq`; MNE objects carry their own.

## A2 — What you can switch

In [ ]:
import inspect
from mne_denoise.asr import ASR, AdaptiveASR, JugglerASR, GuidedASR

def params(cls):
    return {n: p.default for n, p in inspect.signature(cls.__init__).parameters.items()
            if n != "self"}

sets = {c.__name__: params(c) for c in (ASR, AdaptiveASR, JugglerASR, GuidedASR)}
shared = set.intersection(*(set(v) for v in sets.values()))

print(f"{'estimator':<14s} {'params':>7s}   variant-specific knobs")
print("-" * 78)
for name, ps in sets.items():
    own = sorted(set(ps) - shared)
    print(f"{name:<14s} {len(ps):>7d}   {', '.join(own)}")
print(f"\n{len(shared)} parameters are shared by all four — one contract, four calibration"
      f" strategies.")

The package's own decision guide (`docs/asr.rst`):

- **reference-compatible start** — `ASR(method="standard")`, then validate the cutoff
- **robust calibration with a working cutoff** — `ASR(method="riemannian_windowed")`
- **online / streaming BCI** — `AdaptiveASR(variant="psw")` or `"psp"`
- **extreme MoBI / high motion** — `JugglerASR(strategy="gev")` or `"dbscan"`

## A3 — Validated against the MATLAB originals

In [ ]:
# This demo lives in its own repository, so ask the installed package where
# its source tree is rather than assuming a directory layout.
root = du.mne_denoise_root()
if root is None:
    print("mne-denoise is installed from a wheel; parity fixtures ship with the "
          "source checkout only.")
else:
    fixtures = sorted((root / "tests" / "parity" / "matlab_reference").glob("*.mat"))
    tests = sorted((root / "tests" / "parity").glob("test_*.py"))
    print(f"{len(fixtures)} MATLAB reference fixtures, {len(tests)} parity test modules:")
    for t in tests:
        print(f"   {t.name}")

> **Say:** the variants are not reimplementations we hope are right — standard ASR, adaptive
> ASR and the Riemannian backend are each pinned against fixtures generated from the original
> MATLAB code.

## A4 — Where each variant comes from

| Variant | Source | Regime it was built for |
|---|---|---|
| `ASR(method="standard")` | Kothe & Jung 2016; Chang et al. 2020 | transient bursts on ordinary EEG |
| `ASR(method="riemannian_windowed")` | Blum et al. 2019 | calibration windows themselves contaminated |
| `AdaptiveASR(variant=...)` | Tsai et al. | non-stationary recordings, streaming BCI |
| `JugglerASR(strategy=...)` | Kim et al. 2025 | extreme MoBI — 205-channel juggling EEG |

Blum's rASR was developed on 24-channel *mobile* EEG with gyroscope, accelerometer and GPS
streams, indoors and outdoors — not sleep. The sleep ASR toolbox is `dusk2dawn`
(Somervail et al. 2023), a separate lineage that happens to embed Blum's Riemannian option.